In [0]:
# Questo notebook aggiorna la Silver meteo con i dati live del giorno appena concluso.
# Legge i dati raccolti da LiveForecast nella tabella Bronze:
# progetto_meteo.bronze.meteo_streaming
# e poi inserisce quei dati nella tabella Silver:
# progetto_meteo.silver.dati_aggiornati
#
# La tabella Bronze contiene più acquisizioni live nella stessa ora,
# perché LiveForecast viene eseguito ogni 15 minuti.
#
# Il Patcher raggruppa queste acquisizioni per:
# - Citta
# - Regione
# - Area
# - Data
# - Ora
#
# e calcola una singola riga oraria.
#
# In questa fase:
# - Temp_Istantanea diventa Temp_Oraria;
# - Umidita viene mediata;
# - Velocita_Vento viene mediata;
# - Precipitazioni vengono mediate;
# - Temp_Max e Temp_Min vengono recuperate dal dato giornaliero;
# - Timestamp prende l'ultima acquisizione disponibile.
#
# Il notebook controlla anche se il giorno è già presente in Silver,
# così evita duplicazioni in silver.dati_aggiornati.

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F


# Imposto catalogo e tabelle del progetto.
catalogo = "progetto_meteo"

tabella_streaming = f"{catalogo}.bronze.meteo_streaming"
tabella_dati_aggiornati = f"{catalogo}.silver.dati_aggiornati"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Definisco le colonne finali della tabella dati_aggiornati.
# Questo ordine viene usato anche dopo l'aggregazione,
# così la Silver resta coerente con il resto della pipeline.
colonne_dati_aggiornati = [
    "Citta",
    "Regione",
    "Area",
    "Data",
    "Ora",
    "Temp_Max",
    "Temp_Min",
    "Temp_Oraria",
    "Umidita",
    "Velocita_Vento",
    "Precipitazioni",
    "Timestamp"
]


# Calcolo il giorno da consolidare.
# Se il job gira dopo mezzanotte, prende il giorno appena concluso.
oggi = datetime.now(ZoneInfo("Europe/Rome")).date()
giorno_da_consolidare = oggi - timedelta(days=1)

print(f"Giorno da consolidare: {giorno_da_consolidare}")


# Controllo se il giorno è già presente in dati_aggiornati.
# Se esiste già almeno una riga per quel giorno, non appendo nulla.
giorno_gia_presente = (
    spark.table(tabella_dati_aggiornati)
    .filter(F.col("Data") == F.lit(giorno_da_consolidare))
    .limit(1)
    .count() > 0
)


if giorno_gia_presente:

    print(f"Il giorno {giorno_da_consolidare} è già presente in dati_aggiornati. Nessuna riga aggiunta.")


else:

    # Leggo solo i dati streaming del giorno appena concluso.
    df_giorno = (
        spark.table(tabella_streaming)
        .filter(F.col("Data") == F.lit(giorno_da_consolidare))
    )


    # Consolido le acquisizioni live in valori orari.
    # Temp_Istantanea diventa Temp_Oraria tramite media.
    df_giorno_consolidato = (
        df_giorno
        .groupBy("Citta", "Regione", "Area", "Data", "Ora")
        .agg(
            F.first("Temp_Max", ignorenulls=True).alias("Temp_Max"),
            F.first("Temp_Min", ignorenulls=True).alias("Temp_Min"),
            F.round(F.avg("Temp_Istantanea"), 2).alias("Temp_Oraria"),
            F.round(F.avg("Umidita"), 2).alias("Umidita"),
            F.round(F.avg("Velocita_Vento"), 2).alias("Velocita_Vento"),
            F.round(F.avg("Precipitazioni"), 2).alias("Precipitazioni"),
            F.max("Timestamp").alias("Timestamp")
        )
        .select(colonne_dati_aggiornati)
    )


    # Controllo se il giorno appena concluso ha davvero righe da aggiungere.
    righe_da_aggiungere = df_giorno_consolidato.count()


    if righe_da_aggiungere == 0:

        print(f"Nessun dato trovato in meteo_streaming per il giorno {giorno_da_consolidare}.")


    else:

        # Appendo il giorno consolidato alla Silver meteo.
        # Uso append perché dati_aggiornati contiene già lo storico e i giorni precedenti.
        df_giorno_consolidato.write.mode("append").format("delta").saveAsTable(
            tabella_dati_aggiornati
        )


        # Controllo finale.
        # Mostro il giorno consolidato ordinato per città e ora.
        display(
            df_giorno_consolidato
            .orderBy("Citta", "Ora")
        )


        # Stampo un riepilogo finale del patch giornaliero.
        print(f"Patch completata per il giorno {giorno_da_consolidare}.")
        print(f"Righe aggiunte a dati_aggiornati: {righe_da_aggiungere}")